# Data Analysis for PAPI detection and classification dataset

## Imports

In [ ]:
# Imports
import pandas as pd
import os
import shutil
import exifread
import numpy as np
import random

## File path sorting/matching

In [ ]:
# Defining paths for data files
data_master_path = "../../Data/PROJECT1-PAPI/"
data_files = os.listdir(data_master_path)

In [ ]:
print("Data files available:")
for file in data_files:
    print(f" - {file}")

In [ ]:
# Load excel file into dataframe
excel_path = os.path.join(data_master_path, "PAPI_Coords_Fred_DE.xlsx")
PAPI_06_df = pd.read_excel(excel_path, header=1, nrows=4)
PAPI_06_df.head(4)

In [ ]:
PAPI_24_df = pd.read_excel(excel_path, header=9, nrows=4)
PAPI_24_df.head(4)

In [ ]:
# Remove excel file from data_files
data_files.remove("PAPI_Coords_Fred_DE.xlsx")

In [ ]:
# Split folders based on name to new folders matching the PAPI lights in the image
PAPI_06_data = []
PAPI_24_data = []
for filename in data_files:
    if "y06" in filename:
        PAPI_06_data.append(filename)
    else:
        PAPI_24_data.append(filename)

In [ ]:
print(f"PAPI 06 datafiles: {PAPI_06_data}")
print(f"PAPI 24 datafiles: {PAPI_24_data}")

In [ ]:
dest_root_06 = "./dataset/PAPI_06"
os.makedirs(dest_root_06, exist_ok=True)

for file_path in PAPI_06_data:
    src = os.path.join(data_master_path, file_path)
    dst = os.path.join(dest_root_06, os.path.basename(file_path))
    
    shutil.copytree(src, dst, dirs_exist_ok=True)

In [ ]:
dest_root_24 = "./dataset/PAPI_24"
os.makedirs(dest_root_24, exist_ok=True)

for file_path in PAPI_24_data:
    src = os.path.join(data_master_path, file_path)
    dst = os.path.join(dest_root_24, os.path.basename(file_path))
    
    shutil.copytree(src, dst, dirs_exist_ok=True)

## Metadata analysis

In [ ]:
img_path = "dataset\PAPI_06\DJI_202604290007_019_300mRwy06night\DJI_20260429002101_0001_V.JPG"
def read_all_metadata(image_path):
    with open(image_path, "rb") as f:
        tags = exifread.process_file(f, details=True)

    for tag in tags:
        print(f"{tag}: {tags[tag]}")
read_all_metadata(img_path)

## Extracting metadata to dataframe

In [ ]:
def parse_ifd_ratio(value):
    """
    Converts exifread IfdTag (DJI-safe) to float
    """
    try:
        v = value.values

        # Case: single Ratio like 465147/1000
        if hasattr(v, "num") and hasattr(v, "den"):
            return float(v.num) / float(v.den)

        # Case: list containing Ratio
        if isinstance(v, (list, tuple)):
            first = v[0]

            if hasattr(first, "num") and hasattr(first, "den"):
                return float(first.num) / float(first.den)

            return float(first)

        return float(v)

    except Exception:
        return None

In [ ]:
def get_gps_exifread(image_path):
    with open(image_path, 'rb') as f:
        tags = exifread.process_file(f, details=False)

    lat = lon = alt = None

    try:
        lat_ref = tags.get("GPS GPSLatitudeRef")
        lat_val = tags.get("GPS GPSLatitude")
        lon_ref = tags.get("GPS GPSLongitudeRef")
        lon_val = tags.get("GPS GPSLongitude")

        alt_ref = tags.get("GPS GPSAltitudeRef")
        alt_val = tags.get("GPS GPSAltitude")

        # ---- LAT/LON ----
        def to_deg(v):
            d, m, s = v.values
            return (
                float(d.num / d.den) +
                float(m.num / m.den) / 60 +
                float(s.num / s.den) / 3600
            )

        if lat_val and lon_val:
            lat = to_deg(lat_val)
            lon = to_deg(lon_val)

            if lat_ref.values != "N":
                lat = -lat
            if lon_ref.values != "E":
                lon = -lon

        # ---- ALTITUDE (NOW FIXED) ----
        if alt_val:
            alt = parse_ifd_ratio(alt_val)

            if alt_ref.values == 1:
                alt = -alt

    except Exception as e:
        print("ERROR:", e)

    return lat, lon, alt

In [ ]:
rows = []

for root, _, files in os.walk("./dataset/PAPI_06"):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg")):
            path = os.path.join(root, file)

            lat, lon, alt = get_gps_exifread(path)

            rows.append({
                "file": file,
                "path": path,
                "latitude": lat,
                "longitude": lon,
                "altitude": alt
            })

df_PAPI_06_metadata = pd.DataFrame(rows)
df_PAPI_06_metadata.head()

In [ ]:
rows = []

for root, _, files in os.walk("./dataset/PAPI_24"):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg")):
            path = os.path.join(root, file)

            lat, lon, alt = get_gps_exifread(path)

            rows.append({
                "file": file,
                "path": path,
                "latitude": lat,
                "longitude": lon,
                "altitude": alt
            })

df_PAPI_24_metadata = pd.DataFrame(rows)
df_PAPI_24_metadata.head()

In [ ]:
PAPI_06_Height = min(df_PAPI_06_metadata["altitude"].values)
print(f"PAPI_06 Approximate floor height: {PAPI_06_Height}")
PAPI_24_Height = min(df_PAPI_24_metadata["altitude"].values)
print(f"PAPI_24 Approximate floor height: {PAPI_24_Height}")

In [ ]:
# PAPI_06_df["altitude"] = PAPI_06_Height
# PAPI_24_df["altitude"] = PAPI_24_Height
PAPI_06_df["altitude"] = 461.37
PAPI_24_df["altitude"] = 461.37
PAPI_06_df.head()

In [ ]:
PAPI_24_df.head()

## Calculate angle to the PAPI lights

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000  # Earth radius (m)

    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)

    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return R * c

In [ ]:
def compute_angles(drone_row, light_row):
    dist = haversine(
        drone_row["latitude"],
        drone_row["longitude"],
        light_row["latitude"],
        light_row["longitude"]
    )

    dh = drone_row["altitude"] - light_row["altitude"]

    elevation = np.degrees(np.arctan2(dh, dist)) if dist != 0 else 0

    return dist, elevation

In [ ]:
rows = []

for _, drone in df_PAPI_06_metadata.iterrows():
    for _, light in PAPI_06_df.iterrows():

        dist, elev = compute_angles(drone, light)

        rows.append({
            "image": drone["file"],
            "drone_lat": drone["latitude"],
            "drone_lon": drone["longitude"],
            "drone_alt": drone["altitude"],
            "light_lat": light["latitude"],
            "light_lon": light["longitude"],
            "light_alt": light["altitude"],
            "distance_m": dist,
            "elevation_angle_deg": elev
        })

df_angles = pd.DataFrame(rows)
df_angles.head()

In [ ]:
print(max(df_angles["elevation_angle_deg"].values))
print(min(df_angles["elevation_angle_deg"].values))

In [ ]:
rows = []

for _, drone in df_PAPI_24_metadata.iterrows():
    for _, light in PAPI_24_df.iterrows():

        dist, elev = compute_angles(drone, light)

        rows.append({
            "image": drone["file"],
            "drone_lat": drone["latitude"],
            "drone_lon": drone["longitude"],
            "drone_alt": drone["altitude"],
            "light_lat": light["latitude"],
            "light_lon": light["longitude"],
            "light_alt": light["altitude"],
            "distance_m": dist,
            "elevation_angle_deg": elev
        })

df_angles = pd.DataFrame(rows)
df_angles.head()

In [ ]:
print(max(df_angles["elevation_angle_deg"].values))
print(min(df_angles["elevation_angle_deg"].values))

## Full Dataset folder merging and analysis

In [ ]:
# =========================================================
# INPUT / OUTPUT
# =========================================================

SOURCE_ROOT = "PAPI_Full"
OUTPUT_ROOT = "PAPI_Merged"

OUTPUT_IMAGES = os.path.join(OUTPUT_ROOT, "images")
OUTPUT_LABELS = os.path.join(OUTPUT_ROOT, "labels")

os.makedirs(OUTPUT_IMAGES, exist_ok=True)
os.makedirs(OUTPUT_LABELS, exist_ok=True)

# =========================================================
# VALID IMAGE EXTENSIONS
# =========================================================

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

# =========================================================
# COUNTERS
# =========================================================

image_count = 0
label_count = 0

# =========================================================
# WALK THROUGH ENTIRE DATASET
# =========================================================

for root, dirs, files in os.walk(SOURCE_ROOT):

    # -----------------------------------------------------
    # MERGE IMAGES
    # -----------------------------------------------------

    if os.path.basename(root).lower() == "images":

        for file in files:

            if not file.lower().endswith(IMAGE_EXTENSIONS):
                continue

            src_path = os.path.join(root, file)

            # Create unique filename
            parent_folder = os.path.basename(os.path.dirname(root))
            new_name = f"{parent_folder}_{file}"

            dst_path = os.path.join(OUTPUT_IMAGES, new_name)

            shutil.copy2(src_path, dst_path)

            image_count += 1

    # -----------------------------------------------------
    # MERGE LABELS
    # -----------------------------------------------------

    if os.path.basename(root).lower() == "labels":

        for file in files:

            if not file.lower().endswith(".txt"):
                continue

            src_path = os.path.join(root, file)

            # Create matching unique filename
            parent_folder = os.path.basename(os.path.dirname(root))
            new_name = f"{parent_folder}_{file}"

            dst_path = os.path.join(OUTPUT_LABELS, new_name)

            shutil.copy2(src_path, dst_path)

            label_count += 1

# =========================================================
# DONE
# =========================================================

print(f"Copied {image_count} images")
print(f"Copied {label_count} labels")
print("Dataset merge complete.")

## Splitting into training, testing and validation folders

In [ ]:
# =========================================================
# SETTINGS
# =========================================================

SOURCE_DATASET = "PAPI_Merged"

IMAGES_DIR = os.path.join(SOURCE_DATASET, "images")
LABELS_DIR = os.path.join(SOURCE_DATASET, "labels")

OUTPUT_DATASET = "PAPI_Split"

TRAIN_RATIO = 0.70
VALID_RATIO = 0.20
TEST_RATIO  = 0.10

RANDOM_SEED = 42

# =========================================================
# CREATE OUTPUT STRUCTURE
# =========================================================

splits = ["train", "valid", "test"]

for split in splits:

    os.makedirs(
        os.path.join(OUTPUT_DATASET, split, "images"),
        exist_ok=True
    )

    os.makedirs(
        os.path.join(OUTPUT_DATASET, split, "labels"),
        exist_ok=True
    )

# =========================================================
# GET ALL IMAGES
# =========================================================

IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")

all_images = [
    f for f in os.listdir(IMAGES_DIR)
    if f.lower().endswith(IMAGE_EXTENSIONS)
]

# Shuffle reproducibly
random.seed(RANDOM_SEED)
random.shuffle(all_images)

# =========================================================
# CALCULATE SPLITS
# =========================================================

total = len(all_images)

train_end = int(total * TRAIN_RATIO)
valid_end = train_end + int(total * VALID_RATIO)

train_files = all_images[:train_end]
valid_files = all_images[train_end:valid_end]
test_files  = all_images[valid_end:]

print(f"Total images: {total}")
print(f"Train: {len(train_files)}")
print(f"Valid: {len(valid_files)}")
print(f"Test : {len(test_files)}")

# =========================================================
# COPY FILES
# =========================================================

def copy_split(files, split_name):

    for image_file in files:

        image_src = os.path.join(IMAGES_DIR, image_file)

        label_file = image_file.rsplit(".", 1)[0] + ".txt"
        label_src = os.path.join(LABELS_DIR, label_file)

        image_dst = os.path.join(
            OUTPUT_DATASET,
            split_name,
            "images",
            image_file
        )

        label_dst = os.path.join(
            OUTPUT_DATASET,
            split_name,
            "labels",
            label_file
        )

        # Copy image
        shutil.copy2(image_src, image_dst)

        # Copy label if exists
        if os.path.exists(label_src):
            shutil.copy2(label_src, label_dst)

# =========================================================
# EXECUTE SPLITS
# =========================================================

copy_split(train_files, "train")
copy_split(valid_files, "valid")
copy_split(test_files, "test")

print("Dataset split complete.")